In [0]:
# =============================================================================
# Notebook 04 — Experimental Feature Encoding & Full Feature Matrix Assembly
# Objective: Encode categorical and numeric experimental variables from
#            ml_perovskites.master, then join with ml_perovskites.X_intrinsic
#            to produce the full L3 feature matrix (material + experimental).
#
# Design principles:
#   - NO target encoding here — target encoding must be done inside CV folds
#     (Notebooks 07+) to prevent data leakage.
#   - Ordinal/frequency encoding is computed globally here because it carries
#     no target information.
#   - Binary flag columns are passed through as-is.
#   - GroupKFold grouping key (doi) is preserved but NEVER used as a feature.
#   - source_row_id is the join key only — not a feature.
#
# Inputs : ml_perovskites.master, ml_perovskites.X_intrinsic
# Output : ml_perovskites.X_exp  (Delta table, L3 full feature matrix)
#          ml_perovskites.X_material (Delta table, L1 material-only subset)
#          ml_perovskites.X_material_thermo (Delta table, L2 subset — placeholder
#                                            until thermodynamic descriptors are
#                                            added in Notebook 05)
# Seed   : 42
# Author : Bernardo Araldi da Silva | UFSC | 2026
# =============================================================================

# =============================================================================
# SECTION 0 — Install dependencies (run this cell alone first)
# =============================================================================
# No additional installs required for this notebook.

# =============================================================================
# SECTION 1 — Imports and configuration
# =============================================================================

import numpy as np
import pandas as pd
import warnings
from pyspark.sql import functions as F

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Columns that identify rows — must NEVER be used as features
ID_COLS = ["source_row_id", "doi", "base_compound", "Compound_original",
           "formula_reduced", "family_proposed", "domain"]

# Numeric experimental columns present in master
# These measure physical conditions and are used as-is after unit harmonization
NUMERIC_EXP_COLS = [
    "lamp_power_W",          # lamp power in Watts
    "cutoff_nm_final",       # light cut-off wavelength in nm
    "catalyst_mass_g",       # catalyst mass in grams
    "organic_vol_fraction",  # organic fraction of solvent (0–1)
    "thermal_T_C",           # calcination/synthesis temperature in °C
    "thermal_time_h",        # calcination/synthesis time in hours
    "volume_mL",             # reaction volume in mL
]

# Categorical experimental columns to encode
# Strategy: ordinal frequency encoding (rank by count descending, 1 = most common)
# This is leakage-free because it uses only the distribution of inputs, not the target.
CATEGORICAL_EXP_COLS = [
    "cocatalyst_class",      # e.g. Pt, Rh, None, other_metal
    "scavenger_class",       # e.g. methanol, TEOA, Na2S/Na2SO3
    "run_type",              # e.g. liquid_phase, gas_phase
    "prep_class",            # e.g. sol_gel, hydrothermal, solid_state
    "rate_confidence",       # high / medium / low — ordinal
    "cutoff_status",         # visible_only / UV_vis / UV_only
]

# Binary flag columns — passed through directly (already 0/1)
FLAG_COLS_PREFIX = "flag_"

print("Section 1 complete: imports and configuration loaded.")

# =============================================================================
# SECTION 2 — Load master and X_intrinsic tables
# =============================================================================

print("\n--- Loading master table ---")
master_pd = spark.table("ml_perovskites.master").toPandas()
print(f"Master shape       : {master_pd.shape}")
print(f"master columns     : {list(master_pd.columns[:10])} ...")

print("\n--- Loading X_intrinsic table ---")
xint_pd = spark.table("ml_perovskites.X_intrinsic").toPandas()
print(f"X_intrinsic shape  : {xint_pd.shape}")
print(f"X_intrinsic columns: {list(xint_pd.columns[:10])} ...")

# Verify join key alignment
n_int = xint_pd["source_row_id"].nunique()
n_master = master_pd["source_row_id"].nunique()
print(f"\nsource_row_id unique — master: {n_master} | X_intrinsic: {n_int}")

# =============================================================================
# SECTION 3 — Identify available columns in master
# =============================================================================

print("\n--- Checking column availability in master ---")

# Flag columns (binary 0/1)
flag_cols_available = [c for c in master_pd.columns if c.startswith(FLAG_COLS_PREFIX)]
print(f"Flag columns found : {flag_cols_available}")

# Numeric experimental columns
numeric_available = [c for c in NUMERIC_EXP_COLS if c in master_pd.columns]
numeric_missing   = [c for c in NUMERIC_EXP_COLS if c not in master_pd.columns]
print(f"Numeric exp cols available : {numeric_available}")
if numeric_missing:
    print(f"Numeric exp cols NOT found : {numeric_missing}")

# Categorical experimental columns
cat_available = [c for c in CATEGORICAL_EXP_COLS if c in master_pd.columns]
cat_missing   = [c for c in CATEGORICAL_EXP_COLS if c not in master_pd.columns]
print(f"Categorical exp cols available : {cat_available}")
if cat_missing:
    print(f"Categorical exp cols NOT found : {cat_missing}")

# =============================================================================
# SECTION 4 — Numeric experimental features
# =============================================================================

print("\n--- Processing numeric experimental features ---")

exp_num = master_pd[["source_row_id"] + numeric_available].copy()

# Coerce to float (columns may have been read as object due to mixed types)
for col in numeric_available:
    exp_num[col] = pd.to_numeric(exp_num[col], errors="coerce")

# NaN audit before imputation
nan_rates_num = exp_num[numeric_available].isna().mean().sort_values(ascending=False)
print("NaN rates in numeric experimental columns:")
print(nan_rates_num.to_string())

# Imputation strategy:
#   - lamp_power_W, cutoff_nm_final: median imputation (physical measurement, skewed)
#   - catalyst_mass_g, volume_mL: median imputation
#   - thermal_T_C, thermal_time_h: median imputation (many NaN for non-calcined samples)
#   - organic_vol_fraction: fill 0 (absence of organic co-solvent is a legitimate value)

fill_with_zero = ["organic_vol_fraction"]
fill_with_median = [c for c in numeric_available if c not in fill_with_zero]

for col in fill_with_zero:
    if col in exp_num.columns:
        n_filled = exp_num[col].isna().sum()
        exp_num[col] = exp_num[col].fillna(0.0)
        print(f"  {col}: filled {n_filled} NaN with 0.0")

for col in fill_with_median:
    if col in exp_num.columns:
        median_val = exp_num[col].median()
        n_filled = exp_num[col].isna().sum()
        exp_num[col] = exp_num[col].fillna(median_val)
        print(f"  {col}: filled {n_filled} NaN with median={median_val:.4g}")

# Rename with "exp_" prefix to distinguish from material descriptors
rename_num = {c: f"exp_{c}" for c in numeric_available}
exp_num = exp_num.rename(columns=rename_num)

print(f"\nNumeric experimental features shape: {exp_num.shape}")
print(f"Columns: {list(exp_num.columns)}")

# =============================================================================
# SECTION 5 — Categorical experimental features (ordinal frequency encoding)
# =============================================================================

print("\n--- Encoding categorical experimental features ---")

exp_cat = master_pd[["source_row_id"] + cat_available].copy()

# For each categorical column:
#   1. Fill NaN with "unknown"
#   2. Compute frequency rank (most frequent category → rank 1)
#      This is leakage-free and scale-invariant.
#   3. Store the mapping for reference and reproducibility

cat_encoding_maps = {}

for col in cat_available:
    n_nan = exp_cat[col].isna().sum()
    exp_cat[col] = exp_cat[col].fillna("unknown").astype(str).str.strip().str.lower()

    # Frequency rank: rank descending by count (ties get same rank, dense)
    freq = exp_cat[col].value_counts()              # series: value → count
    rank_map = {val: rank+1 for rank, val in enumerate(freq.index)}  # 1 = most common
    cat_encoding_maps[col] = rank_map

    new_col = f"exp_{col}_freq"
    exp_cat[new_col] = exp_cat[col].map(rank_map).fillna(len(rank_map) + 1).astype(int)

    # Also store n_unique flag
    n_unique = exp_cat[col].nunique()
    print(f"  {col}: {n_unique} categories, {n_nan} NaN → filled 'unknown', "
          f"rank range [1, {max(rank_map.values())}]")

# Drop the original string columns — keep only encoded versions
encoded_cols = [f"exp_{c}_freq" for c in cat_available]
exp_cat = exp_cat[["source_row_id"] + encoded_cols]

print(f"\nCategorical encoded features shape: {exp_cat.shape}")
print(f"Columns: {list(exp_cat.columns)}")

# =============================================================================
# SECTION 6 — Binary flag features
# =============================================================================

print("\n--- Collecting binary flag features ---")

if flag_cols_available:
    exp_flags = master_pd[["source_row_id"] + flag_cols_available].copy()

    # Coerce to int (0/1); NaN or non-numeric → 0 (absence of flag = not flagged)
    for col in flag_cols_available:
        exp_flags[col] = (pd.to_numeric(exp_flags[col], errors="coerce")
                          .fillna(0)
                          .astype(float)   # float first so NaN is safe
                          .round()
                          .astype(int))

    print(f"Flag columns shape: {exp_flags.shape}")
    print(f"Columns: {flag_cols_available}")
    print("\nFlag column distributions (sum of 1s):")
    print(exp_flags[flag_cols_available].sum().to_string())
else:
    # Create empty placeholder with just the key
    exp_flags = master_pd[["source_row_id"]].copy()
    print("No flag columns found in master. Proceeding without flag features.")

# =============================================================================
# SECTION 7 — Merge all experimental feature blocks
# =============================================================================

print("\n--- Merging experimental feature blocks ---")

exp_all = exp_num.copy()

if exp_cat.shape[1] > 1:   # has columns beyond source_row_id
    exp_all = exp_all.merge(exp_cat, on="source_row_id", how="left")

if exp_flags.shape[1] > 1:
    exp_all = exp_all.merge(exp_flags, on="source_row_id", how="left")

print(f"Experimental features combined shape: {exp_all.shape}")
print(f"Number of experimental features     : {exp_all.shape[1] - 1}")  # subtract key col

# =============================================================================
# SECTION 8 — Build full L3 feature matrix (X_exp)
# =============================================================================

print("\n--- Building full L3 feature matrix ---")

# Join: X_intrinsic LEFT JOIN experimental features
# All 682 parse_ok rows from X_intrinsic; experimental features filled via master

# Pull provenance columns that exist in master but are NOT already in xint_pd.
# Exclude source_row_id itself (it's the join key, not a provenance column to add).
already_in_xint = set(xint_pd.columns)
prov_cols = [
    c for c in ID_COLS
    if c != "source_row_id"
    and c in master_pd.columns
    and c not in already_in_xint
]
master_prov = master_pd[["source_row_id"] + prov_cols].drop_duplicates("source_row_id")

x_full = xint_pd.merge(exp_all, on="source_row_id", how="left")
if prov_cols:
    x_full = x_full.merge(master_prov, on="source_row_id", how="left")

print(f"Full L3 matrix shape: {x_full.shape}")

# Identify feature columns (exclude id/prov cols and all metadata columns from xint_pd)
# Metadata columns in xint_pd: everything up to and including B_site_elements / n_B_elements
XINT_META_COLS = ["source_row_id", "base_compound", "family_proposed", "formula_reduced",
                  "n_elements", "n_A_elements", "n_B_elements",
                  "A_site_elements", "B_site_elements", "parse_ok"]
non_feature_cols = set(XINT_META_COLS + [c for c in ID_COLS if c in x_full.columns])
feature_cols = [c for c in x_full.columns if c not in non_feature_cols]

# Further exclude metadata columns that leaked from X_intrinsic
# (goldschmidt stats columns are features; ionic_radius_A/B are features)
n_wav         = sum(1 for c in feature_cols if c.startswith("wav_") and not c.startswith("wav_A_") and not c.startswith("wav_B_"))
n_wav_A       = sum(1 for c in feature_cols if c.startswith("wav_A_"))
n_wav_B       = sum(1 for c in feature_cols if c.startswith("wav_B_"))
n_delta       = sum(1 for c in feature_cols if c.startswith("delta_"))
n_structural  = sum(1 for c in feature_cols if c in [
    "goldschmidt_t", "octahedral_mu", "ionic_radius_A", "ionic_radius_B",
    "valence_electrons_B", "electronegativity_diff_AB"])
n_exp_numeric = sum(1 for c in feature_cols if c.startswith("exp_") and "freq" not in c)
n_exp_cat     = sum(1 for c in feature_cols if c.startswith("exp_") and "freq" in c)
n_flags       = sum(1 for c in feature_cols if c.startswith("flag_"))

print(f"\nFeature breakdown:")
print(f"  wav_*       (whole-formula WAV)  : {n_wav}")
print(f"  wav_A_*     (A-site WAV)         : {n_wav_A}")
print(f"  wav_B_*     (B-site WAV)         : {n_wav_B}")
print(f"  delta_*     (A–B deltas)         : {n_delta}")
print(f"  structural                        : {n_structural}")
print(f"  exp_*_freq  (encoded categorical) : {n_exp_cat}")
print(f"  exp_*       (numeric conditions)  : {n_exp_numeric}")
print(f"  flag_*      (binary flags)        : {n_flags}")
print(f"  TOTAL features                    : {len(feature_cols)}")

# NaN audit on full matrix
nan_rates = x_full[feature_cols].isna().mean().sort_values(ascending=False)
high_nan = nan_rates[nan_rates > 0.50]
if len(high_nan) > 0:
    print(f"\n  WARNING: {len(high_nan)} feature columns with >50% NaN:")
    print(high_nan.to_string())
else:
    print("\n  No feature columns with >50% NaN.")

# =============================================================================
# SECTION 9 — Build L1 (material-only) subset
# =============================================================================

print("\n--- Building L1 material-only feature matrix ---")

intrinsic_feature_cols = [c for c in feature_cols
                          if not c.startswith("exp_") and not c.startswith("flag_")]
x_material = x_full[["source_row_id"] + intrinsic_feature_cols].copy()
print(f"L1 material-only shape: {x_material.shape} ({len(intrinsic_feature_cols)} features)")

# =============================================================================
# SECTION 10 — Acceptance criteria checks
# =============================================================================

print("\n" + "="*60)
print("ACCEPTANCE CRITERIA CHECKS")
print("="*60)

# CHECK 1: Row count
n_rows = len(x_full)
print(f"\n[CHECK 1] Row count: {n_rows} (expected 682)")
print("  PASS" if n_rows == 682 else f"  FAIL — got {n_rows}")

# CHECK 2: source_row_id uniqueness
n_uid = x_full["source_row_id"].nunique()
print(f"\n[CHECK 2] source_row_id unique values: {n_uid} / {n_rows}")
print("  PASS" if n_uid == n_rows else "  FAIL — duplicate source_row_ids")

# CHECK 3: No target column in feature set
target_leak_check = [c for c in feature_cols
                     if any(t in c.lower() for t in
                            ["h2", "hydrogen", "rate", "aqy", "quantum_yield",
                             "doi", "source_row"])]
print(f"\n[CHECK 3] Target/ID leakage check: {len(target_leak_check)} suspicious cols")
if target_leak_check:
    print(f"  WARNING: {target_leak_check}")
else:
    print("  PASS — no target or ID columns found in feature set")

# CHECK 4: Experimental features present
n_exp_total = n_exp_numeric + n_exp_cat + n_flags
print(f"\n[CHECK 4] Experimental features: {n_exp_total} total")
print("  PASS" if n_exp_total > 0 else "  FAIL — no experimental features")

# CHECK 5: NaN in key numeric exp columns
key_exp_nan = {col: x_full[f"exp_{col}"].isna().mean()
               for col in numeric_available
               if f"exp_{col}" in x_full.columns}
all_zero_nan = all(v == 0 for v in key_exp_nan.values())
print(f"\n[CHECK 5] NaN in numeric experimental cols after imputation:")
for col, rate in key_exp_nan.items():
    print(f"  exp_{col}: {rate*100:.2f}% NaN")
print("  PASS" if all_zero_nan else "  WARNING — residual NaN found")

print("\n" + "="*60)

# =============================================================================
# SECTION 11 — Write Delta tables
# =============================================================================

print("\n--- Writing ml_perovskites.X_exp as Delta table ---")

x_full_clean = x_full.where(pd.notnull(x_full), other=None)
x_full_spark = spark.createDataFrame(x_full_clean)

(x_full_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.X_exp"))

n_written = spark.table("ml_perovskites.X_exp").count()
print(f"Rows written to ml_perovskites.X_exp: {n_written}")

print("\n--- Writing ml_perovskites.X_material as Delta table ---")

x_material_clean = x_material.where(pd.notnull(x_material), other=None)
x_material_spark = spark.createDataFrame(x_material_clean)

(x_material_spark
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ml_perovskites.X_material"))

n_mat = spark.table("ml_perovskites.X_material").count()
print(f"Rows written to ml_perovskites.X_material: {n_mat}")

# Verify
spark.table("ml_perovskites.X_exp").printSchema()

print("\n" + "="*60)
print("NOTEBOOK 04 COMPLETE")
print("="*60)
print(f"Output tables:")
print(f"  ml_perovskites.X_exp      : {n_written} rows, {x_full.shape[1]} columns")
print(f"  ml_perovskites.X_material : {n_mat} rows, {x_material.shape[1]} columns")
print(f"\nTotal features in L3 (X_exp)     : {len(feature_cols)}")
print(f"  - intrinsic (L1)               : {len(intrinsic_feature_cols)}")
print(f"  - experimental numeric         : {n_exp_numeric}")
print(f"  - experimental categorical enc : {n_exp_cat}")
print(f"  - flags                        : {n_flags}")
print(f"\nJoin key (NOT a feature): source_row_id")
print(f"Grouping key (NOT a feature): doi")
print(f"\nNOTE: Target encoding for categorical variables must be done INSIDE")
print(f"      each CV fold in Notebooks 07+. The frequency-encoded versions")
print(f"      here are safe for any model but may underperform for tree models.")
print(f"\nNext step: Notebook 05 — Target definition, log-transform, and")
print(f"           GroupKFold split strategy validation.")